# India Runs · Track 1 — Redrob Candidate Ranker (Sandbox)

**Submission spec 10.5 sandbox.** Runs the exact ranking system end-to-end on a
small candidate sample → ranked CSV, then checks the CSV is actually present.
CPU-only, no network, no hosted LLM.

## 1. Get the code (always a clean, fresh clone)

In [1]:
import os, shutil

REPO_URL = 'https://github.com/Jakmaroli/india-runs-redrob-ranker'

%cd /content
if os.path.exists('repo'):
    shutil.rmtree('repo')
!git clone -q $REPO_URL repo
%cd /content/repo
!pip -q install -r requirements.txt --break-system-packages 2>/dev/null || pip -q install -r requirements.txt
print('Working directory:', os.getcwd())
print('rank.py present:', os.path.exists('rank.py'))


/content
/content/repo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 17.1 MB/s eta 0:00:00
Working directory: /content/repo
rank.py present: True


## 2. Safety patch (idempotent)

Overwrites `ranker/textmatch.py` with the verified-correct version regardless
of what's currently on GitHub. This guarantees the sandbox runs even if the
repo hasn't been re-synced -- harmless if it's already correct.


In [2]:
FIXED_TEXTMATCH = '"""\ntextmatch.py\n============\nThe semantic half -- "understand the role, not the keywords".\n\nDEFAULT: TF-IDF over candidate career narratives, cosine vs a JD-INTENT query.\nSatisfies the compute contract out of the box (CPU, no network, no downloads).\n\nOPTIONAL (no hosted LLM, no network at rank time): if a precomputed offline\nsentence-transformer index is present, load_embedding_scores loads it from disk and\nscores cosine vs the precomputed JD vector. rank.py prefers it when --cache is given,\nelse falls back to TF-IDF. Nothing here calls any API.\n"""\nfrom __future__ import annotations\nimport os\nfrom typing import Dict, List, Optional\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import linear_kernel\n\nJD_QUERY = (\n    "senior ai engineer building production ranking retrieval and recommendation "\n    "systems. embeddings based retrieval, vector search, hybrid search, semantic "\n    "search, learning to rank, re-ranking. shipped end to end search recommendation "\n    "system to real users at scale in production at a product company. information "\n    "retrieval, nlp, language models, relevance. evaluation of ranking with ndcg mrr "\n    "map offline online a/b testing. python, scalable inference, feature engineering, "\n    "machine learning systems. matching candidates to jobs, talent intelligence."\n)\n\n\ndef candidate_doc(c: Dict) -> str:\n    p = (c.get("profile") or {})\n    # FIX: .get(key, "") only substitutes when the KEY is missing, not when its\n    # value is explicitly None -- a common pattern in real scraped/aggregated\n    # data. Without the `or ""` guard, a single None field crashes " ".join()\n    # with TypeError and aborts the entire run. Reproduced directly: a\n    # candidate with current_title=None took down the whole 100k-row pipeline.\n    parts = [p.get("headline") or "", p.get("summary") or "",\n             p.get("current_title") or "", p.get("current_industry") or ""]\n    for h in c.get("career_history", []) or []:\n        seg = f\'{h.get("title") or ""} {h.get("description") or ""} {h.get("industry") or ""}\'\n        parts += [seg, seg]\n    parts.append(" ".join((s.get("name") or "") for s in (c.get("skills", []) or [])))\n    return " ".join(parts)\n\n\ndef _minmax(sims: np.ndarray) -> np.ndarray:\n    lo, hi = float(sims.min()), float(sims.max())\n    if hi - lo < 1e-9:\n        return np.zeros_like(sims)\n    return (sims - lo) / (hi - lo)\n\n\ndef _fit_tfidf(docs: List[str], min_df: int):\n    vec = TfidfVectorizer(\n        sublinear_tf=True, max_features=30000, ngram_range=(1, 2),\n        min_df=min_df, stop_words="english", dtype=np.float32,\n    )\n    cand_mat = vec.fit_transform(docs)\n    query_vec = vec.transform([JD_QUERY])\n    return cand_mat, query_vec\n\n\ndef semantic_scores(docs: List[str]) -> np.ndarray:\n    # min_df=5 is tuned for the full 100K pool. On small/diverse samples\n    # (sandbox tests, demos) requiring a term in >=5 of a handful of very\n    # different career narratives can prune the entire vocabulary to zero,\n    # which raises a ValueError either way ("max_df corresponds to < docs\n    # than min_df" or "no terms remain after pruning"). Use min_df=5 only\n    # once the corpus is large enough that the threshold is meaningful, and\n    # fall back to min_df=1 -- both on small corpora and defensively if the\n    # tuned threshold ever empties the vocabulary anyway.\n    n_docs = len(docs)\n    min_df = 5 if n_docs >= 50 else 1\n    try:\n        cand_mat, query_vec = _fit_tfidf(docs, min_df)\n    except ValueError:\n        cand_mat, query_vec = _fit_tfidf(docs, 1)\n    sims = linear_kernel(query_vec, cand_mat).ravel()\n    return _minmax(sims)\n\n\ndef load_embedding_scores(cache_dir: str, candidate_ids: List[str]) -> Optional[np.ndarray]:\n    ids_p = os.path.join(cache_dir, "candidate_ids.npy")\n    emb_p = os.path.join(cache_dir, "embeddings.npy")\n    jd_p = os.path.join(cache_dir, "jd_embedding.npy")\n    if not all(os.path.exists(p) for p in (ids_p, emb_p, jd_p)):\n        return None\n    ids = np.load(ids_p, allow_pickle=True).tolist()\n    embs = np.load(emb_p)\n    jd = np.load(jd_p)\n    sims = embs @ jd\n    sim_map = {cid: float(s) for cid, s in zip(ids, sims)}\n    aligned = np.array([sim_map.get(cid, 0.0) for cid in candidate_ids], dtype=np.float32)\n    return _minmax(aligned)\n'

with open('ranker/textmatch.py', 'w', encoding='utf-8') as f:
    f.write(FIXED_TEXTMATCH)

print("ranker/textmatch.py patched.")
print([l for l in FIXED_TEXTMATCH.splitlines() if "min_df" in l])


ranker/textmatch.py patched.
['def _fit_tfidf(docs: List[str], min_df: int):', '        min_df=min_df, stop_words="english", dtype=np.float32,', '    # min_df=5 is tuned for the full 100K pool. On small/diverse samples', '    # than min_df" or "no terms remain after pruning"). Use min_df=5 only', '    # fall back to min_df=1 -- both on small corpora and defensively if the', '    min_df = 5 if n_docs >= 50 else 1', '        cand_mat, query_vec = _fit_tfidf(docs, min_df)']


## 3. Candidate sample (6 candidates: a clear top performer, a clear keyword-stuffer trap, and a few in between)

In [3]:
import json

DEMO_JSON = '[{"candidate_id": "CAND_0000001", "profile": {"anonymized_name": "A", "headline": "Senior ML Engineer | Search & Ranking", "summary": "7 yrs building production ranking and retrieval systems at product companies; shipped recommendation engine at scale.", "location": "Pune", "country": "India", "years_of_experience": 7, "current_title": "Senior ML Engineer", "current_company": "ProductCo", "current_company_size": "201-500", "current_industry": "Software"}, "career_history": [{"company": "ProductCo", "title": "Senior ML Engineer", "start_date": "2021-01-01", "end_date": null, "duration_months": 54, "is_current": true, "industry": "Software", "company_size": "201-500", "description": "Built semantic search and learning-to-rank recommendation system; NDCG offline eval and A/B testing; deployed to 1M users."}], "education": [{"institution": "IIT", "degree": "B.Tech", "field_of_study": "CS", "start_year": 2012, "end_year": 2016, "grade": "8.5", "tier": "tier_1"}], "skills": [{"name": "Semantic Search", "proficiency": "expert", "endorsements": 20, "duration_months": 48}, {"name": "FAISS", "proficiency": "advanced", "endorsements": 10, "duration_months": 24}, {"name": "Python", "proficiency": "expert", "endorsements": 30, "duration_months": 84}], "redrob_signals": {"last_active_date": "2026-06-10", "open_to_work_flag": true, "recruiter_response_rate": 0.9, "notice_period_days": 15, "willing_to_relocate": true, "verified_email": true, "github_activity_score": 70, "interview_completion_rate": 0.9, "skill_assessment_scores": {"Python": 90, "NLP": 85}}}, {"candidate_id": "CAND_0000002", "profile": {"anonymized_name": "B", "headline": "HR Manager | NLP | RAG | Embeddings", "summary": "HR professional interested in AI.", "location": "Mumbai", "country": "India", "years_of_experience": 6, "current_title": "HR Manager", "current_company": "BigCorp", "current_company_size": "10001+", "current_industry": "Consulting"}, "career_history": [{"company": "BigCorp", "title": "HR Manager", "start_date": "2019-01-01", "end_date": null, "duration_months": 77, "is_current": true, "industry": "Consulting", "company_size": "10001+", "description": "Managed recruitment and HR operations."}], "education": [{"institution": "Local U", "degree": "MBA", "field_of_study": "HR", "start_year": 2013, "end_year": 2015, "grade": "7", "tier": "tier_3"}], "skills": [{"name": "NLP", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "RAG", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "Embeddings", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "FAISS", "proficiency": "expert", "endorsements": 0, "duration_months": 0}], "redrob_signals": {"last_active_date": "2026-01-01", "open_to_work_flag": false, "recruiter_response_rate": 0.1, "notice_period_days": 90, "willing_to_relocate": false, "verified_email": true}}, {"candidate_id": "CAND_0000003", "profile": {"anonymized_name": "C", "headline": "Data Scientist | Search Relevance", "summary": "5 yrs improving search relevance and ranking models at a marketplace company.", "location": "Bengaluru", "country": "India", "years_of_experience": 5, "current_title": "Data Scientist", "current_company": "MarketCo", "current_company_size": "501-1000", "current_industry": "E-commerce"}, "career_history": [{"company": "MarketCo", "title": "Data Scientist", "start_date": "2020-06-01", "end_date": null, "duration_months": 72, "is_current": true, "industry": "E-commerce", "company_size": "501-1000", "description": "Improved search relevance ranking; ran offline NDCG evaluation and online A/B tests."}], "education": [{"institution": "BITS Pilani", "degree": "B.E.", "field_of_study": "CS", "start_year": 2011, "end_year": 2015, "grade": "8.0", "tier": "tier_1"}], "skills": [{"name": "Python", "proficiency": "advanced", "endorsements": 15, "duration_months": 60}, {"name": "Search Relevance", "proficiency": "advanced", "endorsements": 8, "duration_months": 40}], "redrob_signals": {"last_active_date": "2026-05-01", "open_to_work_flag": true, "recruiter_response_rate": 0.6, "notice_period_days": 30, "willing_to_relocate": true, "verified_email": true}}, {"candidate_id": "CAND_0000004", "profile": {"anonymized_name": "D", "headline": "Marketing Manager | Vector DB | RAG | Embeddings", "summary": "Marketing professional with interest in AI tools.", "location": "Delhi", "country": "India", "years_of_experience": 8, "current_title": "Marketing Manager", "current_company": "AdCo", "current_company_size": "1001-5000", "current_industry": "Advertising"}, "career_history": [{"company": "AdCo", "title": "Marketing Manager", "start_date": "2018-01-01", "end_date": null, "duration_months": 90, "is_current": true, "industry": "Advertising", "company_size": "1001-5000", "description": "Led brand campaigns and marketing analytics dashboards."}], "education": [{"institution": "Local U", "degree": "MBA", "field_of_study": "Marketing", "start_year": 2008, "end_year": 2010, "grade": "6.5", "tier": "tier_3"}], "skills": [{"name": "Pinecone", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "RAG", "proficiency": "expert", "endorsements": 0, "duration_months": 0}, {"name": "Vector Search", "proficiency": "expert", "endorsements": 0, "duration_months": 0}], "redrob_signals": {"last_active_date": "2025-11-01", "open_to_work_flag": false, "recruiter_response_rate": 0.05, "notice_period_days": 90, "willing_to_relocate": false, "verified_email": true}}, {"candidate_id": "CAND_0000005", "profile": {"anonymized_name": "E", "headline": "Software Engineer | IT Services", "summary": "Backend developer at an IT services firm, mostly client maintenance projects.", "location": "Noida", "country": "India", "years_of_experience": 6, "current_title": "Software Engineer", "current_company": "ServiceCo", "current_company_size": "10001+", "current_industry": "IT Services"}, "career_history": [{"company": "ServiceCo", "title": "Software Engineer", "start_date": "2019-01-01", "end_date": null, "duration_months": 78, "is_current": true, "industry": "IT Services", "company_size": "10001+", "description": "Maintained client CRUD applications and bug fixes for outsourced projects."}], "education": [{"institution": "Local College", "degree": "B.Tech", "field_of_study": "IT", "start_year": 2009, "end_year": 2013, "grade": "7.2", "tier": "tier_2"}], "skills": [{"name": "Java", "proficiency": "advanced", "endorsements": 10, "duration_months": 70}], "redrob_signals": {"last_active_date": "2026-04-01", "open_to_work_flag": true, "recruiter_response_rate": 0.5, "notice_period_days": 60, "willing_to_relocate": true, "verified_email": true}}, {"candidate_id": "CAND_0000006", "profile": {"anonymized_name": "F", "headline": "ML Engineer | Frequent Job Changes", "summary": "ML engineer who has changed companies frequently.", "location": "Hyderabad", "country": "India", "years_of_experience": 5, "current_title": "ML Engineer", "current_company": "StartupX", "current_company_size": "11-50", "current_industry": "Software"}, "career_history": [{"company": "StartupX", "title": "ML Engineer", "start_date": "2025-09-01", "end_date": null, "duration_months": 9, "is_current": true, "industry": "Software", "company_size": "11-50", "description": "Built ML pipelines."}, {"company": "StartupY", "title": "ML Engineer", "start_date": "2024-09-01", "end_date": "2025-08-01", "duration_months": 11, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Built recommendation prototypes."}, {"company": "StartupZ", "title": "ML Engineer", "start_date": "2023-09-01", "end_date": "2024-08-01", "duration_months": 11, "is_current": false, "industry": "Software", "company_size": "11-50", "description": "Built search prototypes."}], "education": [{"institution": "Anna University", "degree": "B.E.", "field_of_study": "CS", "start_year": 2014, "end_year": 2018, "grade": "7.8", "tier": "tier_2"}], "skills": [{"name": "Python", "proficiency": "advanced", "endorsements": 5, "duration_months": 50}], "redrob_signals": {"last_active_date": "2026-06-01", "open_to_work_flag": true, "recruiter_response_rate": 0.7, "notice_period_days": 30, "willing_to_relocate": true, "verified_email": true}}]'

demo = json.loads(DEMO_JSON)
with open('sample_candidates.jsonl', 'w', encoding='utf-8') as f:
    f.write('\n'.join(json.dumps(c) for c in demo))

print('sample ready:', sum(1 for _ in open('sample_candidates.jsonl')), 'candidates')


sample ready: 6 candidates


## 4. Run the ranker (CPU, no network) -- fails loudly instead of lying about timing

In [4]:
import time, subprocess, os

t = time.time()
result = subprocess.run(
    ['python', 'rank.py', '--candidates', 'sample_candidates.jsonl', '--out', 'demo_submission.csv'],
    capture_output=True, text=True,
)
elapsed = time.time() - t

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f'rank.py failed with exit code {result.returncode} after {elapsed:.1f}s -- see traceback above')

csv_exists = os.path.exists('demo_submission.csv')
print(f'ranked in {elapsed:.1f}s (budget 5 min)')
print('CSV PRESENT:', csv_exists)
assert csv_exists, 'rank.py exited 0 but demo_submission.csv is missing -- stop and report this'



[1/6] Loading candidates from sample_candidates.jsonl ...
      loaded 6 (0.0s)
[2/6] Semantic relevance to the JD intent ...
      using TF-IDF semantic scores
      semantic ready (0.1s)
[3/6] features + [4/6] traps + [5/6] scoring ...
      scored all; flagged 0 honeypots (0.1s)
[6/6] Top-100 -> demo_submission.csv ...
Done. Wrote 6 rows in 0.1s. Honeypots in top-100: 0.

ranked in 1.8s (budget 5 min)
CSV PRESENT: True


## 4b. Download the demo CSV to your own computer

This is the small demo file (6 fake candidates) -- proof the pipeline runs end to end. It is **not** your real submission; that's the `submission.csv` built from the actual 100K-candidate file.


In [5]:
# Actually download the CSV to your computer (not just leave it in the Colab VM)
try:
    from google.colab import files
    files.download('demo_submission.csv')
except ImportError:
    print('Not running in Colab -- demo_submission.csv is in the current working directory:', __import__('os').getcwd())


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Ranked shortlist (plain `csv` module -- avoids numpy/pandas ABI conflicts entirely)

In [6]:
import csv

with open('demo_submission.csv', newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

print(f'{len(rows)} rows in demo_submission.csv\n')
for r in rows:
    print(f"rank {r['rank']:>2}  score {float(r['score']):.3f}  {r['candidate_id']}")
    print(f"    {r['reasoning'][:160]}")


6 rows in demo_submission.csv

rank  1  score 1.000  CAND_0000001
    Senior ML Engineer with 7.0 yrs (~4 in applied ML/data roles); built ranking/retrieval work as Senior ML Engineer at ProductCo; covers 4/4 must-have skill areas
rank  2  score 0.716  CAND_0000003
    Data Scientist with 5.0 yrs (~6 in applied ML/data roles); built ranking/retrieval work as Data Scientist at MarketCo; covers 3/4 must-have skill areas; strong 
rank  3  score 0.353  CAND_0000006
    ML Engineer with 5.0 yrs (~3 in applied ML/data roles); closest relevant role: ML Engineer at StartupY; strong on Python; recently active, 70% recruiter respons
rank  4  score 0.187  CAND_0000005
    Software Engineer with 6.0 yrs; 50% recruiter response, open to work; based in Noida.
rank  5  score 0.000  CAND_0000002
    HR Manager with 6.0 yrs; strong on NLP, Embeddings, FAISS; 10% recruiter response; based in Mumbai. Concern: AI skills listed but career is non-engineering, 90-
rank  6  score 0.000  CAND_0000004
    Marke

## 6. Validate format + honeypot rate

Note: the `100 rows` / `ranks 1..100` checks are expected to **fail** here --
this is intentionally checking on a 6-candidate demo, not the real
100-candidate submission. What matters for the sandbox is everything else:
duplicates, score ordering, and the honeypot rate.

In [7]:
import subprocess
r = subprocess.run(['python', 'validate_submission.py', '--submission', 'demo_submission.csv',
                     '--candidates', 'sample_candidates.jsonl'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)


Format checks:
  [PASS] header == candidate_id,rank,score,reasoning
  [FAIL] exactly 100 data rows
  [FAIL] ranks 1..100 each exactly once
  [PASS] candidate_ids unique
  [PASS] every candidate_id exists in pool
  [PASS] score non-increasing with rank
  [PASS] scores not all identical
  [PASS] no empty reasoning
  [PASS] reasonings not all identical
Honeypot check (Stage-3 DQ if >10% of top-100):
  [PASS] honeypot rate = 0.0% (<= 10%)

VALIDATION FAILED -- fix the above before uploading.




---
**Summary:** the strong engineer ranks #1; the keyword-stuffed, unavailable
profiles correctly sink to the bottom with score 0. No LLM, no GPU, no
network -- reproducible in the Stage-3 sandbox.

In [8]:
print("="*50)
print("SANDBOX RESULT: PASS" if csv_exists and result.returncode == 0 else "SANDBOX RESULT: FAIL")
print("="*50)


SANDBOX RESULT: PASS
